# 📦 Notebook 2: Delivery Semantics + Dead Letter Queues

When a message goes from producer → broker → consumer, what guarantees do we get?

There are three options people talk about:

- 🟥 **At-most-once** — fire and forget. Message *may* be lost. **Never duplicated.**
- 🟨 **At-least-once** — keep retrying until the consumer says "got it!" Message *may be duplicated*. Never lost.
- 🟩 **Exactly-once** — what we wish we had. Hard and expensive in distributed systems; usually faked with at-least-once + idempotency.

We'll simulate all three with a flaky in-memory broker, then add a **dead letter queue (DLQ)** for messages that keep failing.

## Learning objectives
- Feel the difference between losing messages and duplicating them.
- Implement an idempotent consumer using a dedup set.
- Use a DLQ to quarantine "poison" messages.

## 🟥 At-most-once: fire and forget

The producer sends and forgets. If the consumer crashes mid-processing, the message is gone forever.

In [ ]:
import random
random.seed(0)

def send_at_most_once(msg, consumer):
    # No ack, no retry. If consumer raises, we lose the message.
    try:
        consumer(msg)
    except Exception as e:
        print(f"  💀 lost {msg}: {e}")

processed = []
def flaky_consumer(msg):
    if random.random() < 0.4:                 # 40% failure rate
        raise RuntimeError("boom")
    processed.append(msg)

for i in range(10):
    send_at_most_once(f"msg-{i}", flaky_consumer)

print("processed:", processed)
print(f"delivered {len(processed)}/10")

## 🟨 At-least-once: retry until ack'd

Now the consumer must explicitly **ack**. If we don't see an ack within a short time (or get an explicit nack), we re-deliver. Result: nothing is lost, but **duplicates can happen** if the consumer crashed *after* doing the work but *before* acking.

In [ ]:
def send_at_least_once(msg, consumer, max_attempts=5):
    for attempt in range(1, max_attempts + 1):
        try:
            consumer(msg)
            return True   # ack
        except Exception:
            print(f"  🔁 retry {attempt} for {msg}")
    return False

processed = []
attempts_seen = []
def sometimes_double_processes(msg):
    attempts_seen.append(msg)
    if random.random() < 0.5:
        # consumer crashed AFTER doing work but BEFORE ack -> retry sees it again
        processed.append(msg)
        raise RuntimeError("crashed after work")
    processed.append(msg)

for i in range(5):
    send_at_least_once(f"msg-{i}", sometimes_double_processes)

print("processed:", processed)
print("(notice the duplicates — that's at-least-once)")

## 🟩 Exactly-once-ish: at-least-once + idempotent consumer

We can't usually get true exactly-once delivery, but we can get **exactly-once *processing***. Trick: every message gets a unique id. The consumer remembers ids it has already handled and skips duplicates.

In [ ]:
seen_ids = set()
results = []

def idempotent_consumer(msg):
    msg_id, payload = msg["id"], msg["payload"]
    if msg_id in seen_ids:
        # already processed; just ack and move on
        return
    seen_ids.add(msg_id)
    if random.random() < 0.4:
        # crash after recording the id? In a real system you'd persist the id
        # in the same DB transaction as the side effect to avoid that hole.
        seen_ids.discard(msg_id)
        raise RuntimeError("oops")
    results.append(payload)

for i in range(5):
    send_at_least_once({"id": f"id-{i}", "payload": f"job-{i}"}, idempotent_consumer)

print("results:", results)
print("(no duplicates even though we retried)")

### Where does the dedup set live in real life?

In this notebook the `seen_ids` set lives in process memory — fine for a demo, useless across restarts. Real systems put the dedup state somewhere **shared and durable**:

- **Redis `SET` / `SETNX`** with a TTL — fast, simple, but the TTL must be longer than the longest possible retry window.
- **Database row with a unique constraint on `message_id`** — the *insert itself* fails on a duplicate, so you also get atomicity with the side effect.
- **Bloom filter** — cheap memory, but probabilistic (false positives = silent drops). Use only when occasional duplicate processing is fine.

The golden rule: **persist the dedup id in the same transaction as the side effect.** Otherwise you can crash *between* recording the id and doing the work, and lose the message anyway.

## 📤 Producer side: the Transactional Outbox pattern

Idempotent consumers fix the *receiver* side. But what about the *sender*? If your service writes to its database **and** publishes a message, what happens if the DB commit succeeds and the publish fails (or vice-versa)? You've got an inconsistency.

The **outbox pattern** fixes this with one trick: in the same DB transaction that does the business write, also insert a row into an `outbox` table. A separate background worker reads `outbox` and publishes to the broker, marking rows as sent.

```
  ┌──────────────┐  one txn   ┌──────────┐
  │ orders table │◀──────────▶│ outbox   │  ← unsent messages
  └──────────────┘            └────┬─────┘
                                   │ poll / CDC
                                   ▼
                              ┌──────────┐
                              │  broker  │
                              └──────────┘
```

Pair this with idempotent consumers and you get end-to-end at-least-once with no lost or extra messages — the practical replacement for "exactly-once."

## ☠️ Dead Letter Queue

What about messages that *always* fail — bad data, a bug in the consumer, a poisoned payload? Retrying them forever blocks healthy traffic. Solution: after N failed attempts, move the message to a **dead letter queue** for a human (or a different process) to look at.

In [ ]:
main_queue = ["job-0", "job-1", "POISON", "job-3", "POISON", "job-5"]
dlq = []

def consumer(msg):
    if "POISON" in msg:
        raise ValueError("malformed")
    print(f"  ✅ processed {msg}")

MAX_ATTEMPTS = 3
for msg in main_queue:
    for attempt in range(1, MAX_ATTEMPTS + 1):
        try:
            consumer(msg)
            break
        except Exception as e:
            if attempt == MAX_ATTEMPTS:
                print(f"  ☠️  giving up on {msg} -> DLQ")
                dlq.append({"msg": msg, "error": str(e)})
            else:
                print(f"  🔁 retry {attempt} for {msg}")

print("\nDLQ contents:", dlq)

## ✅ Recap

- **At-most-once** = simple but lossy. Use only when losing data is okay (metrics samples, etc.).
- **At-least-once** = the practical default. Combine with idempotent consumers.
- **Exactly-once** = mostly marketing. Get the same effect with at-least-once + dedup keys.
- **DLQ** = your safety valve for messages that won't process. Always set a DLQ in production.